# Client API in Depth

Learn the contract between your local computation and an NVIDIA FLARE workflow. By the end, you should be able to write a client loop, explain what its messages contain, report meaningful metrics, and select a process mode.

This lesson uses a tiny NumPy linear model so you can focus on the API. **PyTorch conversion and Lightning trainer patching are covered in their own lessons**; the lifecycle and model-exchange principles here apply to both.

## What you will run

Two simulated sites fit a three-parameter linear model to synthetic data generated locally. Each site has 64 training rows and 32 separate validation rows. Sites use different seeds and slightly different feature distributions. There is no dataset download or GPU requirement.

| File | Responsibility |
| --- | --- |
| `prepare_data.py` | Generate each site's reproducible training and validation data |
| `model.py` | Local gradient descent and mean squared error (MSE) |
| `client.py` | Receive, compute, and send with the Client API |
| `job.py` | Configure `NumpyFedAvgRecipe` and the simulation |

Data generation here simulates independent sites on one machine. In a deployment, each client loads its own local data.

## Install lesson dependencies

First complete the [course environment setup](../02.0_introduction/introduction.ipynb) and select its kernel. NVFLARE is installed there; this lesson's requirements add only NumPy.

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import Code, display

# Make notebook shell commands use the selected kernel's environment.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")
lesson = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "client_api_in_depth.ipynb").is_file())
code_dir = lesson / "code"
print("Python:", sys.executable)
print("Example code:", code_dir)
%cd {code_dir}

In [ ]:
%cd {code_dir}
!python -m pip install -r requirements.txt
assert _exit_code == 0, "Command failed; see the output above."

## Understand the client lifecycle

The recipe owns the server's rounds and aggregation. The client owns local data loading, evaluation, and training.

```text
initialize once
  └─ wait for a task → receive its model → compute locally → send one result
       ↑___________________________________________________________|
finish when the job ends; release Client API resources
```

| API | Meaning in this example |
| --- | --- |
| `flare.init()` | Connect the script to its NVFLARE Client API environment; call once |
| `flare.is_running()` | Wait for work and check whether the job can continue; not a round counter |
| `flare.receive()` | Get the current task's `FLModel` |
| `flare.get_site_name()` | Identify the site whose local data to use |
| `flare.get_task_name()` / `flare.is_train()` | Identify the current task; this example supports training |
| `flare.send(result)` | Return one result for the received task; normally clears the current-model cache |
| `flare.shutdown()` | Release Client API resources when the script finishes |

In in-process mode, `is_running()` can wait for and cache the next model. The following `receive()` retrieves that model; it does not start another round. Calling `receive()` repeatedly before sending can return the same cached model. After `send()`, the next iteration waits for the next task. The server decides how many rounds to run and when the job ends.

`receive(timeout=...)` can return `None` when no model arrives before its timeout or the session ends. Check for `None` before using its parameters. Do not treat a receive timeout as a successful training result.

## Read the `FLModel` contract

`FLModel` describes a message; it does not train a model or prescribe one universal parameter layout.

| Field | Meaning | This example |
| --- | --- | --- |
| `params` | Parameters or another payload understood by the workflow | `{NPConstants.NUMPY_KEY: weights}` with a three-element NumPy array |
| `params_type` | Whether the payload is full parameters (`FULL`) or parameter differences (`DIFF`) | Client constructs a `FULL` result |
| `metrics` | Named evaluation measurements interpreted by the workflow | `{"mse": global_mse}` |
| `current_round`, `total_rounds` | Round context supplied by the server | Zero-based current round; printed as round + 1 |
| `meta` | Workflow metadata, including aggregation weight information | Number of local gradient steps |
| `optimizer_params` | Optional optimizer state carried by a workflow that supports it | Unused |

`NumpyFedAvgRecipe` initializes the global vector from `[0.0, 0.0, 0.0]`. Its NumPy persistor uses `NPConstants.NUMPY_KEY` (`"numpy_key"`) as the dictionary key. A different recipe or custom workflow can use a different structure: match its contract rather than assuming every received model has a `"weights"` key.

Keep parameter names and shapes compatible across clients. Do not mutate the received global array while training: preserve it as the starting point and, for difference transfer, the base of the update.

## Keep local computation independent of communication

The data helper creates disjoint training and validation rows. The model helper copies the received weights and performs a configurable number of full-batch gradient steps. These functions know nothing about NVFLARE.

In [ ]:
display(Code(filename=str(code_dir / "prepare_data.py"), language="python"))
display(Code(filename=str(code_dir / "model.py"), language="python"))

## Separate global evaluation, local training, and state

For each task:

1. Evaluate the **received global weights** on local validation data.
2. Train a copy of those weights on local training data.
3. Optionally evaluate the **locally trained weights** for your own diagnostics.
4. Send the trained weights, the **global-model validation metric**, and the local step count.

When an `FLModel` contains both parameters and metrics, NVFLARE's conversion attaches those metrics as `INITIAL_METRICS`. The standard model selector interprets them as evaluation of the received global model. Sending post-training local loss in that field would compare different local models instead. In this example, `local_mse` is printed separately; only `global_mse` is sent as `metrics["mse"]`. The recipe sets `key_metric="mse"` and `key_metric_mode="min"` because lower MSE is better.

The standard FedAvg workflow uses `FLMetaKey.NUM_STEPS_CURRENT_ROUND` to weight contributions, together with any configured site aggregation weights. Here both sites report five full-batch steps and have equal-sized training datasets, so their weights are equal. The step count is **not automatically the number of training samples**. With unequal datasets or local work, choose the aggregation policy deliberately.

Load data once outside the loop, but start every local training round from the newly received global weights. This example uses stateless gradient descent. With momentum or Adam, explicitly decide whether local optimizer state is reset or retained; exchanging model weights does not automatically define that policy.

The default `send()` can clear the received and outgoing `FLModel.params` buffers after serialization. Compute diagnostics before sending, or retain independent copies if you need them later. Avoid `clear_cache=False` in an ordinary one-result-per-task loop: retaining the cached input can make the next iteration process the same task again.

## Implement the client loop

The script below is the complete client implementation you will run. It checks the task type, evaluates before training, constructs the expected payload, and sends one result. `finally` releases API resources even if local computation raises an exception.

In [ ]:
display(Code(filename=str(code_dir / "client.py"), language="python"))

## Connect the client to a recipe

The recipe supplies the initial model, server-side averaging, and persistence. The client always returns full weights; the recipe can optionally configure automatic difference transfer. The external trainer uses `python -u` from the active environment. The notebook setup puts the selected kernel’s environment first on `PATH`; terminal users should activate that same environment.

The three-round default is deliberately short. All supporting modules live in this lesson's `code/` directory and travel with the exported application.

In [ ]:
display(Code(filename=str(code_dir / "job.py"), language="python"))

## Run the default in-process job

Run cells in order. The command uses the selected kernel's environment and stops this notebook on a nonzero exit code.

Look for three rounds, two client contributions per round, and a decreasing global `mse`. That metric evaluates the model **at the beginning** of each round. The saved model includes the final round's update, so the last reported pre-training MSE is not a fresh evaluation of the final checkpoint.

`log_config="progress"` keeps the console focused on progress and the run summary. Detailed client messages, including `received`, `global_mse`, and `local_mse`, remain in each site's `log.txt`.

In [ ]:
%cd {code_dir}
!python job.py
assert _exit_code == 0, "Command failed; see the output above."

## Inspect what the server saved

The checkpoint is the three-element global vector, not a client optimizer or dataset. Its order is the two feature coefficients followed by the bias. The data-generating coefficients are `[2.0, -1.0, 0.5]`; a short run should move toward them without being expected to recover them exactly.

In [ ]:
import numpy as np

workspace = Path("/tmp/nvflare/chapter2/client_api")
result = workspace / "client_api_in_process_full" / "server" / "simulate_job"
print("Final global weights:", np.load(result / "models" / "server.npy"))
print("Client log:", workspace / "client_api_in_process_full" / "site-1" / "log.txt")

## Choose who owns the trainer process

| Mode | Who starts the trainer? | When to use it | Configuration |
| --- | --- | --- | --- |
| `in_process` | NVFLARE runs the script inside the Client Job process | Simple local development and lightweight training | Recipe `launch_external_process=False` |
| `external_process` | NVFLARE starts and manages a separate trainer | Separate process lifecycle, process isolation, or a distributed training launcher | Recipe `launch_external_process=True`; set `command` for the launcher |
| `attach` | Another system starts and owns the trainer | An independently managed trainer must connect to an NVFLARE job | Configure `ClientAPIExecutor(execution_mode="attach")` and the trainer connection explicitly |

Compare the first two modes below using the **same client script**. The launcher changes process ownership; it does not rewrite local training or partition your data for you. External process mode alone does not configure multi-GPU training. Attach setup and distributed training details belong in the [advanced Attach guide](https://nvflare.readthedocs.io/en/main/programming_guide/execution_api_type/client_api_attach.html).

In [ ]:
%cd {code_dir}
!python job.py --execution_mode external_process
assert _exit_code == 0, "Command failed; see the output above."

## Compare full and difference transfer

For a received global vector `w` and trained local vector `w_local`:

- `FULL` represents `w_local`.
- `DIFF` represents `w_local - w`.

Our client constructs `ParamsType.FULL` in both runs. With `params_transfer_type=TransferType.DIFF`, the Client API computes the difference against the received model before transmitting it, and the matching recipe workflow applies the aggregated update. This is why local training must not change the received array in place.

If a custom client computes a difference itself, it must label that payload `ParamsType.DIFF` and use a compatible workflow. The Client API respects that label rather than differencing an already labelled DIFF again. Do not send a delta labelled FULL.

The next run changes only transfer configuration. With the same data and local work, the learned weights should be close to the full-transfer run, allowing for floating-point differences.

In [ ]:
%cd {code_dir}
!python job.py --transfer_type diff
assert _exit_code == 0, "Command failed; see the output above."

## Export the same application

`--export` is handled by the Recipe API and exports the application without training. There is no local `--export` declaration in `job.py`. The exported configuration contains the selected process mode and parameter-transfer settings.

In [ ]:
%cd {code_dir}
!python job.py --export --export-dir /tmp/nvflare/chapter2/exports/client_api
assert _exit_code == 0, "Command failed; see the output above."

## Try a controlled change

Change the first run command to:

```python
!python job.py --local_steps 1 --workspace_dir /tmp/nvflare/chapter2/client_api_one_step
assert _exit_code == 0, "Command failed; see the output above."
```

Before running, predict the outcome:

- Should the first round's global MSE change when both runs start from zero weights and use the same validation rows?
- Should the local MSE and later rounds change when clients do fewer gradient steps?
- If one site does more local work, how would the reported step counts affect its aggregation weight?

For a second exercise, compare the saved vectors from `client_api_in_process_full`, `client_api_external_process_full`, and `client_api_in_process_diff`. Explain why process ownership and wire representation should not change the intended learning algorithm.

## Troubleshoot the client contract

| Symptom | What to check |
| --- | --- |
| `receive()` appears to wait | Waiting between tasks is normal. If it never advances, inspect the server and client logs to confirm the job started and assigned a task; running `client.py` alone does not create a server |
| Server waits for a contribution | Check that every supported task reaches one `send()`, and look for a local training exception |
| Same round repeats | Check for multiple receives before sending or retained input from `clear_cache=False` |
| Parameter key or shape error | Compare the recipe's initial-model contract with the client's received and returned payloads |
| DIFF calculation fails or results jump unexpectedly | Preserve the received global parameters, label full/difference results correctly, and match the recipe's transfer configuration |
| MSE changes but model selection is confusing | Verify which model the metric evaluates and whether `key_metric_mode` should be `min` or `max` |
| Imports fail only in notebook commands | Compare the printed kernel Python with the terminal environment; rerun setup and use the same environment |
| A cell fails and old results still exist | Stop and fix the failed command. The exit-status assertion prevents later cells from silently reading stale checkpoints |

For additional APIs, see the [Client API reference](https://nvflare.readthedocs.io/en/main/apidocs/nvflare.client.api.html).

## Continue to framework-specific conversion

The Client API contract is shared across frameworks. Continue to [PyTorch Lightning](../02.3_convert_torch_lightning_to_federated_learning/convert_torch_lightning_to_fl.ipynb), or revisit [plain PyTorch conversion](../../chapter-1_running_federated_learning_applications/01.2_convert_deep_learning_to_federated_learning/convert_dl_to_fl.ipynb) for its training-loop integration.

Use the Client API with a built-in recipe when the server workflow already fits your application. When you need to write custom coordination as published Python method calls, see the [Collab API lesson](../02.6_collab_api/collab_api.ipynb).